# ENSTA — Séance 1 · Observer, représenter, apprendre

**Introduction à l’apprentissage profond · dernière année · septembre 2026**  
**Version étudiante · exercices à compléter · Python / PyTorch · processeur CPU suffisant**

Ce notebook accompagne la première séance consacrée à l’apprentissage à partir de données. Son fil directeur est volontairement méthodique : **avant de construire un modèle, il faut regarder les observations ; avant d’interpréter un score, il faut comprendre le protocole qui l’a produit ; avant de faire confiance à un gradient, il faut savoir quelle quantité il dérive.**

Nous commencerons donc par le **quartet d’Anscombe**. Quatre jeux de données possèdent des résumés numériques presque identiques, mais leurs représentations graphiques racontent des histoires radicalement différentes. Cette entrée en matière servira à reprendre les vecteurs, les matrices, les axes de calcul et les statistiques descriptives, tout en installant une règle qui restera valable pendant toute la séance :

> **prévoir → exécuter → observer → expliquer.**

La suite conserve les cinq séquences pratiques du cours. Un problème de classification de type XOR permettra de distinguer **représentation**, **optimisation** et **généralisation**. Un oscillateur amorti montrera enfin comment une équation différentielle peut devenir une source d’information pour un réseau. Ce dernier exemple est un cas d’étude contrôlé : il ne cherche pas à concurrencer un solveur classique, mais à rendre visibles la construction et les limites d’un PINN.

| Séquence pratique | Repère dans la séance de 5 h | Temps | Question centrale |
|---|---:|---:|---|
| [Prélude — quartet d’Anscombe](#anscombe) | 00:00–00:20 | 20 min | Que perd-on lorsqu’on résume sans représenter ? |
| [A — Tenseurs et protocole](#tp-a) | 00:40–01:00 | 20 min | Comment préparer les données sans fuite ? |
| [B — Linéaire ou non linéaire ?](#tp-b) | 01:25–01:55 | 30 min | Quand faut-il apprendre une représentation ? |
| [C — Ouvrir la boîte du gradient](#tp-c) | 02:35–03:05 | 30 min | Que calcule exactement la rétropropagation ? |
| [D — Choisir sans regarder le test](#tp-d) | 03:30–03:55 | 25 min | Comment protéger une conclusion expérimentale ? |
| [E — L’oscillateur comme contrainte](#tp-e) | 04:25–04:55 | 30 min | Comment une loi physique devient-elle une perte ? |
| [Ticket de sortie](#sortie) | 04:55–05:00 | 5 min | Relier données, modèle, gradient et validation |

Les plages intermédiaires sont consacrées au cours et aux deux pauses. Dans chaque partie, commencez par le **socle obligatoire** ; les prolongements sont explicitement signalés comme facultatifs. Travaillez en binôme : une personne tient le clavier, l’autre annonce les dimensions attendues, formule une prédiction et vérifie que le résultat répond bien à la question posée.


## Démarrage — à effectuer avant la séance si possible

Ouvrez le notebook dans JupyterLab, VS Code ou Google Colab. Dans Colab :

1. importez le fichier ou ouvrez-en une copie dans votre Drive ;
2. choisissez un environnement **Python 3 / CPU** ;
3. exécutez les cellules dans l’ordre ;
4. sauvegardez régulièrement votre copie, car l’état d’exécution du runtime est temporaire.

Un GPU n’est nécessaire pour aucune expérience de cette séance. Les cellules marquées `TODO` s’arrêtent volontairement avec `NotImplementedError` tant que la réponse n’a pas été écrite : ce comportement ne signale pas une panne. Les assertions placées après les exercices constituent des contrôles locaux ; elles vérifient une propriété précise, mais ne remplacent ni l’observation des figures ni l’interprétation des résultats.

La commande d’installation de la cellule suivante est désactivée. Ne la décommentez que si un import échoue. Une fois les bibliothèques chargées, l’ensemble du TP fonctionne sans téléchargement de données. La visualisation Plotly du quartet d’Anscombe est facultative : si Plotly n’est pas disponible, la figure Matplotlib demeure suffisante pour réaliser l’activité.


In [ ]:
# À décommenter seulement si nécessaire, puis redémarrer le noyau.
# %pip install torch numpy matplotlib scipy


In [ ]:
import copy
import math
import platform
import time
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 2026
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)  # petits tenseurs : limiter le surcoût du parallélisme
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
NOTEBOOK_START = time.perf_counter()

def export_figure(fig, name):
    # Option pour l'enseignant : définir ENSTA_FIGURE_DIR pour exporter les figures.
    destination = os.environ.get("ENSTA_FIGURE_DIR")
    if destination:
        directory = Path(destination)
        directory.mkdir(parents=True, exist_ok=True)
        fig.savefig(directory / f"{name}.pdf", bbox_inches="tight")
        fig.savefig(directory / f"{name}.png", dpi=160, bbox_inches="tight")
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for index, axis in enumerate(fig.axes, start=1):
            bbox = axis.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted()).expanded(1.05, 1.10)
            # Masquer les axes voisins pour que leurs textes ne débordent pas dans le recadrage.
            visibility = [other.get_visible() for other in fig.axes]
            for other in fig.axes:
                other.set_visible(other is axis)
            fig.savefig(directory / f"{name}_panel{index}.pdf", bbox_inches=bbox)
            fig.savefig(directory / f"{name}_panel{index}.png", dpi=160, bbox_inches=bbox)
            for other, was_visible in zip(fig.axes, visibility):
                other.set_visible(was_visible)
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | {DEVICE}")
print("Les graines fixées facilitent la comparaison ; une identité bit à bit entre machines n'est pas garantie.")


<a id="anscombe"></a>
## Prélude — Le quartet d’Anscombe : voir les données avant de les modéliser · 20 min

En 1973, le statisticien Francis Anscombe a construit quatre petits jeux de données pour montrer une idée simple et durable : **des résumés numériques presque identiques peuvent correspondre à des structures très différentes**. Le but n’est pas de dévaloriser la moyenne, la variance, la corrélation ou la régression linéaire. Ces quantités répondent à des questions précises. Le danger apparaît lorsqu’on leur demande de remplacer l’examen de la structure des observations.

Chaque jeu contient 11 couples $(x_i,y_i)$. Nous représenterons donc :

- un jeu par une matrice de forme `(11, 2)` ;
- les quatre jeux par un tenseur de forme `(4, 11, 2)` ;
- l’axe 0 comme l’indice du jeu, l’axe 1 comme l’indice de l’observation et l’axe 2 comme la variable $x$ ou $y$.

Cette convention fournit une première lecture des tenseurs : une dimension n’est pas un simple nombre, elle possède une **signification expérimentale**.


In [ ]:
ANSCOMBE = torch.tensor([
    [[10., 8.04], [8., 6.95], [13., 7.58], [9., 8.81], [11., 8.33],
     [14., 9.96], [6., 7.24], [4., 4.26], [12., 10.84], [7., 4.82], [5., 5.68]],
    [[10., 9.14], [8., 8.14], [13., 8.74], [9., 8.77], [11., 9.26],
     [14., 8.10], [6., 6.13], [4., 3.10], [12., 9.13], [7., 7.26], [5., 4.74]],
    [[10., 7.46], [8., 6.77], [13., 12.74], [9., 7.11], [11., 7.81],
     [14., 8.84], [6., 6.08], [4., 5.39], [12., 8.15], [7., 6.42], [5., 5.73]],
    [[8., 6.58], [8., 5.76], [8., 7.71], [8., 8.84], [8., 8.47],
     [8., 7.04], [8., 5.25], [19., 12.50], [8., 5.56], [8., 7.91], [8., 6.89]],
], dtype=torch.float64)
ANSCOMBE_LABELS = ("I", "II", "III", "IV")

print("Forme du tenseur complet :", tuple(ANSCOMBE.shape))
print("Un jeu de données :", tuple(ANSCOMBE[0].shape))
print("Une observation :", tuple(ANSCOMBE[0, 0].shape), "->", ANSCOMBE[0, 0].tolist())


### 0.1 — Indexer un tenseur en donnant un sens à chaque axe · 4 min

Avant d’exécuter le code, annoncez les formes attendues. Extrayez :

1. la matrice complète du premier jeu ;
2. toutes les valeurs de $x$ des quatre jeux ;
3. le vecteur des valeurs de $y$ du troisième jeu.

Utilisez exclusivement l’indexation des tenseurs. L’exercice est élémentaire, mais il installe une habitude fondamentale : **écrire la forme attendue avant l’opération**.


In [ ]:
# TODO 0.1 — remplacer les trois valeurs None par des indexations de ANSCOMBE.
jeu_I = None
tous_les_x = None
y_jeu_III = None

assert jeu_I.shape == (11, 2)
assert tous_les_x.shape == (4, 11)
assert y_jeu_III.shape == (11,)
print("jeu_I :", tuple(jeu_I.shape))
print("tous_les_x :", tuple(tous_les_x.shape))
print("y_jeu_III :", tuple(y_jeu_III.shape))


### 0.2 — Résumer les quatre jeux par un même calcul vectorisé · 7 min

Pour chaque jeu $j$, calculez sur l’axe des observations :

- les moyennes $\bar x_j$ et $\bar y_j$ ;
- les variances d’échantillon $s_{x,j}^2$ et $s_{y,j}^2$, avec le dénominateur $n-1$ ;
- la covariance et la corrélation de Pearson ;
- la pente $a_j=\operatorname{cov}(x,y)/s_x^2$ et l’ordonnée à l’origine $b_j=\bar y_j-a_j\bar x_j$ de la droite des moindres carrés.

La fonction doit recevoir un tenseur de forme `(J, N, 2)` et rendre une matrice `(J, 7)`. Évitez une boucle sur les observations : PyTorch sait agréger un axe entier. Une boucle sur les quatre jeux serait acceptable, mais le calcul vectorisé rend mieux visible le rôle des dimensions.


In [ ]:
def summarize_anscombe(data):
    """Retourne [mean_x, mean_y, var_x, var_y, corr, slope, intercept] pour chaque jeu."""
    # TODO 0.2 — compléter le calcul vectorisé sur l'axe des observations.
    raise NotImplementedError("Compléter summarize_anscombe")

anscombe_stats = summarize_anscombe(ANSCOMBE)
assert anscombe_stats.shape == (4, 7)
assert torch.isfinite(anscombe_stats).all()


In [ ]:
columns = ("moy. x", "moy. y", "var. x", "var. y", "corr.", "pente", "intercept")
print("jeu | " + " | ".join(f"{name:>9s}" for name in columns))
print("-" * 88)
for label, row in zip(ANSCOMBE_LABELS, anscombe_stats):
    print(f" {label:>2s} | " + " | ".join(f"{value.item():9.4f}" for value in row))

# Les nombres ne sont pas exactement identiques à cause de l'arrondi des données originales,
# mais ils sont suffisamment proches pour conduire au même résumé verbal.
assert torch.max(torch.abs(anscombe_stats[:, 0] - 9.0)) < 1e-12
assert torch.max(torch.abs(anscombe_stats[:, 1] - 7.5)) < 1e-3
assert torch.max(torch.abs(anscombe_stats[:, 5] - 0.5)) < 5e-4


### 0.3 — Représenter sur les mêmes axes · 6 min

Le tableau suggère quatre relations presque interchangeables : mêmes moyennes, mêmes variances, corrélations proches et quasiment la même droite de régression. Nous allons maintenant confronter ce résumé aux observations elles-mêmes.

Les quatre panneaux utilisent exactement les mêmes limites d’axes. Cette précaution est importante : des échelles différentes pourraient créer ou masquer visuellement des contrastes. La droite superposée est celle calculée à partir de chaque jeu ; elle n’est pas réajustée pour « mieux suivre » la forme observée.


In [ ]:
def plot_anscombe(data, stats):
    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    x_line = torch.linspace(3.0, 20.0, 200, dtype=data.dtype)

    for j, (label, ax) in enumerate(zip(ANSCOMBE_LABELS, axes.flat)):
        x = data[j, :, 0]
        y = data[j, :, 1]
        slope, intercept = stats[j, 5], stats[j, 6]
        ax.scatter(x.numpy(), y.numpy(), s=42, edgecolor="white", linewidth=0.8)
        ax.plot(x_line.numpy(), (slope * x_line + intercept).numpy(), linewidth=1.6)
        for i, (xi, yi) in enumerate(zip(x, y)):
            ax.annotate(str(i), (xi.item(), yi.item()), xytext=(4, 3),
                        textcoords="offset points", fontsize=7, alpha=0.7)
        ax.set_title(f"Jeu {label} — r = {stats[j, 4].item():.3f}")
        ax.set_xlim(3, 20)
        ax.set_ylim(2, 14)
        ax.grid(alpha=0.25)
        ax.set_xlabel("x")
        ax.set_ylabel("y")

    fig.suptitle("Quartet d’Anscombe — mêmes résumés, structures différentes", fontsize=13)
    export_figure(fig, "anscombe_quartet")
    plt.show()
    return fig

anscombe_figure = plot_anscombe(ANSCOMBE, anscombe_stats)


In [ ]:
# Visualisation interactive facultative : survolez les points, zoomez et comparez les panneaux.
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    print("Plotly n'est pas installé : la figure Matplotlib suffit pour poursuivre.")
else:
    fig_interactive = make_subplots(
        rows=2, cols=2,
        subplot_titles=[f"Jeu {label}" for label in ANSCOMBE_LABELS],
        shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=0.09, vertical_spacing=0.12,
    )
    x_line_np = np.linspace(3.0, 20.0, 200)
    for j, label in enumerate(ANSCOMBE_LABELS):
        row, col = divmod(j, 2)
        row, col = row + 1, col + 1
        x_np = ANSCOMBE[j, :, 0].numpy()
        y_np = ANSCOMBE[j, :, 1].numpy()
        slope = anscombe_stats[j, 5].item()
        intercept = anscombe_stats[j, 6].item()
        fig_interactive.add_trace(
            go.Scatter(
                x=x_np, y=y_np, mode="markers+text",
                text=[str(i) for i in range(len(x_np))], textposition="top center",
                customdata=np.arange(len(x_np)),
                hovertemplate="point %{customdata}<br>x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>",
                name=f"Jeu {label}", showlegend=False,
            ), row=row, col=col,
        )
        fig_interactive.add_trace(
            go.Scatter(
                x=x_line_np, y=slope * x_line_np + intercept,
                mode="lines", hoverinfo="skip", showlegend=False,
            ), row=row, col=col,
        )
    fig_interactive.update_xaxes(range=[3, 20], title_text="x")
    fig_interactive.update_yaxes(range=[2, 14], title_text="y")
    fig_interactive.update_layout(
        height=650, width=900,
        title="Quartet d’Anscombe — exploration interactive",
        template="plotly_white",
    )
    fig_interactive.show()


### 0.4 — Décrire avant d’expliquer · 3 min

Rédigez vos observations sans utiliser les mots « bizarre » ou « normal » : nommez la structure qui apparaît.

1. **Jeu I :** quelle relation simple semble raisonnable, et quelle dispersion subsiste ?
2. **Jeu II :** quelle propriété de la relation est invisible dans la corrélation et la droite ?
3. **Jeu III :** quel point attire l’attention, et comment pourrait-on étudier son influence sans le supprimer arbitrairement ?
4. **Jeu IV :** que se passerait-il pour la variance de $x$ et la régression si le point d’abscisse 19 était absent ?
5. Formulez en une phrase la règle méthodologique à transporter vers les jeux de données de grande dimension, où l’on ne peut plus tout afficher directement.

**Transition.** Le quartet ne montre pas que les statistiques sont inutiles. Il montre qu’un résumé n’épuise pas la structure des données. Dans le TP A, nous conserverons cette exigence d’observation tout en construisant un protocole adapté à un échantillon plus grand.


<a id="tp-a"></a>
## TP A — Des observations à un protocole expérimental · 20 min

Le quartet d’Anscombe vient de montrer qu’un tableau de statistiques descriptives ne suffit pas à caractériser la structure d’un jeu de données. Nous allons maintenant passer d’une petite collection que l’on peut inspecter entièrement à un échantillon plus grand, destiné à l’apprentissage. La difficulté n’est plus seulement de **voir** les données : il faut organiser une expérience qui permette de distinguer ce que le modèle ajuste, ce qui sert à le choisir et ce qui restera réservé à l’évaluation finale.

### Question de départ

Les entrées sont deux coordonnées tirées uniformément dans $[-1.6,1.6]^2$. La classe vaut 1 lorsque leur produit est positif, puis chaque étiquette est inversée avec une probabilité de 8 %. Il s’agit d’une géométrie de type XOR, avec échange des noms de classes : les coordonnées de même signe portent ici l’étiquette 1.

Le bruit d’étiquette joue un rôle essentiel. Même un oracle connaissant parfaitement la règle géométrique ne peut prédire les inversions aléatoires. Son exactitude moyenne est donc de 92 %, mais un échantillon fini ne contiendra pas nécessairement exactement 8 % d’étiquettes inversées. Une erreur observée peut provenir du bruit, d’une famille de modèles insuffisante, d’un apprentissage imparfait ou simplement de la fluctuation d’échantillonnage : ces causes devront rester distinctes.

**Protocole fixé avant l’apprentissage :** 900 observations, réparties en 540 exemples d’entraînement, 180 de validation et 180 de test. Le partage aléatoire est acceptable ici parce que les observations sont indépendantes et issues de la même distribution par construction. Pour des séries temporelles, plusieurs images d’un même objet ou des mesures regroupées par sujet, ce même découpage pourrait créer une fuite et donner une vision excessivement favorable de la généralisation.


In [ ]:
def make_xor(n=900, seed=SEED, flip_probability=0.08):
    g = torch.Generator().manual_seed(seed)
    x = 3.2 * torch.rand(n, 2, generator=g) - 1.6
    y_clean = (x[:, 0] * x[:, 1] > 0).long()
    flips = torch.rand(n, generator=g) < flip_probability
    y = torch.logical_xor(y_clean.bool(), flips).long()
    return x, y

X_all, y_all = make_xor()
g_split = torch.Generator().manual_seed(SEED + 1)
indices = torch.randperm(len(X_all), generator=g_split)
id_train, id_val, id_test = indices[:540], indices[540:720], indices[720:]
X_train_raw, y_train = X_all[id_train], y_all[id_train]
X_val_raw, y_val = X_all[id_val], y_all[id_val]
# Le test est scellé : aucun score, graphique ou réglage avant la fin de D.
TEST_SCELLE = (X_all[id_test].clone(), y_all[id_test].clone())
del X_all, y_all
assert set(id_train.tolist()).isdisjoint(id_val.tolist())
assert set(id_train.tolist()).isdisjoint(id_test.tolist())
assert set(id_val.tolist()).isdisjoint(id_test.tolist())
print("Apprentissage / validation / test :", len(id_train), len(id_val), len(id_test))
print("Équilibre des classes (train seulement) :", torch.bincount(y_train).tolist())

fig, ax = plt.subplots(figsize=(5.2, 4.1))
ax.scatter(X_train_raw[:, 0], X_train_raw[:, 1], c=y_train, cmap="coolwarm", s=12, alpha=.7)
ax.set(xlabel="$x_1$", ylabel="$x_2$", title="Échantillon d'apprentissage : XOR bruité", aspect="equal")
export_figure(fig, "classification_data")
plt.show()


### A1 — Lire une couche affine dans ses dimensions · 5 min

Après avoir manipulé une matrice de données de forme $(11,2)$ dans le prélude, nous considérons maintenant un mini-lot $X$ de forme $(N,d)$. Dans cette question, la matrice des poids est volontairement stockée sous la forme $W\in\mathbb{R}^{d\times C}$ et le biais sous la forme $b\in\mathbb{R}^{C}$. Les scores s’écrivent

$$Z=XW+b,$$

et doivent avoir la forme $(N,C)$. Complétez la fonction sans boucle sur les observations. L’objectif n’est pas seulement d’obtenir un résultat : annoncez d’abord la forme de chaque objet, puis vérifiez que le calcul obtenu respecte cette prédiction.

**À expliquer.** Pourquoi le même biais est-il ajouté à chacune des $N$ lignes ? Quelle forme PyTorch utilise-t-il pour `nn.Linear(d, C).weight` ? Cette convention interne est transposée par rapport à celle de l’exercice ; savoir passer de l’une à l’autre évite de nombreuses erreurs silencieuses.


In [ ]:
def affine(x, w, b):
    # TODO A1 : renvoyer les scores de toutes les observations.
    raise NotImplementedError("A1 : produit matriciel et diffusion du biais")


In [ ]:
a = torch.tensor([[1., 2.], [3., 4.], [-1., 2.]])
w = torch.tensor([[1., 0., -1.], [2., 1., 0.]])
b = torch.tensor([0.5, -0.5, 1.])
z = affine(a, w, b)
assert z.shape == (3, 3)
assert torch.allclose(z[0], torch.tensor([5.5, 1.5, 0.]))
print("Forme des scores :", tuple(z.shape))
print("Stockage nn.Linear(2, 3).weight :", tuple(nn.Linear(2, 3).weight.shape))


### A2 — Transformer les données sans consulter l’avenir · 10 min

Une transformation de données possède elle-même des paramètres. Pour centrer et réduire les deux coordonnées, nous devons estimer une moyenne et un écart-type ; ces quantités font donc partie du modèle expérimental. Elles seront calculées **sur l’ensemble d’entraînement seulement**, colonne par colonne, puis appliquées sans modification à la validation et, plus tard, au test.

Conservez une dimension singleton, de façon que la diffusion des dimensions soit visible dans le code. Nous utiliserons `unbiased=False` pour l’écart-type descriptif et `clamp_min(1e-6)` pour traiter explicitement une coordonnée éventuellement constante.

**À expliquer.** Il est normal que les données de validation transformées n’aient pas exactement une moyenne nulle ni un écart-type égal à un : elles constituent un autre échantillon. Leur retirer leur propre moyenne utiliserait une information indisponible au moment où le modèle est déployé et modifierait la procédure que l’on prétend évaluer.


In [ ]:
def fit_standardizer(x_train):
    # TODO A2 : calculer deux tenseurs de forme (1, d).
    raise NotImplementedError("A2 : moyenne et écart-type sur le train uniquement")

mean_train, std_train = fit_standardizer(X_train_raw)
X_train = (X_train_raw - mean_train) / std_train
X_val = (X_val_raw - mean_train) / std_train


In [ ]:
assert mean_train.shape == std_train.shape == (1, 2)
assert torch.allclose(X_train.mean(0), torch.zeros(2), atol=1e-6)
assert torch.allclose(X_train.std(0, unbiased=False), torch.ones(2), atol=1e-6)
assert y_train.dtype == torch.long
print("Moyenne train après transformation :", X_train.mean(0).tolist())
print("Moyenne validation après transformation :", X_val.mean(0).tolist())
print("Entrées :", X_train.dtype, tuple(X_train.shape), "| cibles :", y_train.dtype, tuple(y_train.shape))


### Bilan du TP A — formuler ce que le code a établi

Rédigez deux ou trois phrases par réponse. Une notation correcte ne suffit pas : reliez le calcul à la logique expérimentale.

1. Quelles sont les dimensions de $X$, $W$, $b$ et $Z$, et comment le biais est-il diffusé ?
2. Pourquoi les statistiques de normalisation appartiennent-elles à l’apprentissage ? Où apparaîtrait une fuite de données ?
3. Quelle hypothèse autorise ici un partage aléatoire, et dans quels types de données faudrait-il la remettre en cause ?


<a id="tp-b"></a>
## TP B — De la frontière affine à la représentation apprise · 30 min

Le protocole étant fixé, nous pouvons maintenant poser une question différente : **la famille de fonctions choisie est-elle capable de représenter la structure observée ?** Une optimisation ne peut pas trouver une fonction qui n’appartient pas à cette famille. La comparaison d’un modèle affine et d’un réseau multicouche isolera cette difficulté de représentation.

### B1 — Deux logits pour une décision binaire · 8 min

Un classifieur à deux classes peut produire deux scores réels, ou **logits**. Le softmax les transforme ensuite en probabilités, mais `CrossEntropyLoss` reçoit directement les logits et effectue une évaluation numériquement stable de la log-softmax et de la perte. **N’ajoutez donc pas de softmax dans le modèle.** Les cibles sont les indices de classes 0 ou 1, de forme $(N,)$ et de type `torch.long`.

Construisez un MLP $2\to32\to32\to2$, avec `Tanh` entre les couches affines et aucune activation en sortie. Le modèle linéaire fourni ne peut produire qu’une frontière droite. Le MLP, lui, apprend des coordonnées intermédiaires dans lesquelles une décision simple pourra devenir possible. `Tanh` n’est pas présentée comme une activation universellement supérieure : elle convient à ce petit problème et sera réutilisée dans le PINN, où la régularité des dérivées joue un rôle explicite.


In [ ]:
def make_mlp(width=32):
    # TODO B1 : trois couches Linear, deux activations Tanh, aucun Softmax.
    raise NotImplementedError("B1 : architecture 2 → width → width → 2")


In [ ]:
probe = make_mlp()
assert probe(X_train[:7]).shape == (7, 2)
assert not any(isinstance(m, nn.Softmax) for m in probe.modules())
print(probe)
print("Paramètres entraînables :", sum(p.numel() for p in probe.parameters()))


### Moteur d’entraînement fourni — comprendre son contrat avant de l’utiliser

La fonction suivante est fournie afin que cette première comparaison porte d’abord sur les **familles de fonctions**, et non sur l’écriture de la boucle d’optimisation. Elle :

- ajuste les paramètres à partir des mini-lots de l’ensemble d’entraînement ;
- recalcule les pertes complètes en mode évaluation ;
- conserve une **copie indépendante** des poids correspondant à la meilleure perte de validation ;
- restaure cette copie avant de rendre le modèle.

Elle ne reçoit aucun jeu de test. Cette absence n’est pas un détail d’interface : elle protège le rôle du test dans l’expérience.

`model.train()` et `model.eval()` règlent le comportement de certaines couches, tandis que `torch.no_grad()` contrôle l’enregistrement des opérations nécessaires au calcul des gradients. Ces deux mécanismes répondent à des questions distinctes. Le MLP de cette partie ne contient pas de dropout, mais nous gardons une fonction d’évaluation correcte et réutilisable.

**Décision annoncée avant les résultats.** Le TP D comparera quatre candidats : le modèle linéaire, le MLP court de B, puis deux MLP entraînés plus longtemps, avec ou sans `weight_decay`. La sélection se fera exclusivement à partir de la perte de validation. Ce protocole reste une petite expérience pédagogique ; il n’a pas la portée d’une étude exhaustive ni d’une validation imbriquée.


In [ ]:
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate_classifier(model, x, y):
    model.eval()
    logits = model(x)
    return {"loss": float(loss_fn(logits, y)),
            "accuracy": float((logits.argmax(dim=1) == y).float().mean())}

def train_classifier(model, x_train, y_train, x_val, y_val,
                     epochs=180, lr=0.01, weight_decay=0.0, seed=SEED + 2):
    model = model.to(DEVICE)
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64,
                        shuffle=True, generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    best_loss, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        train_metrics = evaluate_classifier(model, x_train, y_train)
        val_metrics = evaluate_classifier(model, x_val, y_val)
        for split, metrics in [("train", train_metrics), ("val", val_metrics)]:
            for key, value in metrics.items():
                history[f"{split}_{key}"].append(value)
        if val_metrics["loss"] < best_loss:
            best_loss, best_epoch = val_metrics["loss"], epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return {"model": model, "history": history, "best_val_loss": best_loss,
            "best_epoch": best_epoch, "epochs": epochs,
            "weight_decay": weight_decay}

def show_histories(experiments, figure_name="classification_learning"):
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.5))
    for name, result in experiments.items():
        h = result["history"]
        epochs = np.arange(1, len(h["train_loss"]) + 1)
        line = axes[0].plot(epochs, h["train_loss"], label=name + " train")[0]
        axes[0].plot(epochs, h["val_loss"], "--", color=line.get_color(), label=name + " val")
        axes[1].plot(epochs, h["val_accuracy"], label=name)
    axes[0].set(xlabel="Époque", ylabel="Entropie croisée", title="Perte : apprentissage / validation")
    axes[1].set(xlabel="Époque", ylabel="Exactitude", title="Validation seulement", ylim=(.35, 1.0))
    for ax in axes:
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.tight_layout()
    export_figure(fig, figure_name)
    plt.show()

def show_boundaries(experiments):
    grid_axis = torch.linspace(-1.8, 1.8, 130)
    gx, gy = torch.meshgrid(grid_axis, grid_axis, indexing="xy")
    grid_raw = torch.stack((gx.ravel(), gy.ravel()), dim=1)
    grid = (grid_raw - mean_train) / std_train
    fig, axes = plt.subplots(1, len(experiments), figsize=(5.1 * len(experiments), 4.0), squeeze=False)
    for ax, (name, result) in zip(axes[0], experiments.items()):
        model = result["model"]
        model.eval()
        with torch.no_grad():
            probability = model(grid).softmax(dim=1)[:, 1].reshape(gx.shape)
        im = ax.contourf(gx, gy, probability, levels=np.linspace(0, 1, 15), cmap="coolwarm", vmin=0, vmax=1)
        ax.contour(gx, gy, probability, levels=[.5], colors="black", linewidths=1)
        ax.scatter(X_val_raw[:, 0], X_val_raw[:, 1], c=y_val, cmap="coolwarm", s=14, edgecolors="white", linewidths=.35)
        ax.set(title=name + " — points de validation", xlabel="$x_1$", ylabel="$x_2$", aspect="equal")
    fig.colorbar(im, ax=axes.ravel().tolist(), label=r"$p_\theta(y=1\mid x)$", shrink=.8)
    export_figure(fig, "classification_boundaries")
    plt.show()


### B2 — Prévoir la géométrie, entraîner, puis expliquer · 17 min

Avant toute exécution, dessinez la forme de frontière que vous attendez pour chaque modèle. Demandez-vous ensuite quel résultat pourrait distinguer un défaut de représentation d’un défaut d’optimisation. Une mauvaise exactitude du modèle affine n’indique pas nécessairement que la descente de gradient fonctionne mal : elle peut simplement révéler que sa frontière est trop contrainte.

Lancez les deux entraînements, puis observez conjointement :

1. la perte d’entraînement et la perte de validation ;
2. la frontière de décision ;
3. la localisation des erreurs ;
4. l’époque retenue par validation.

Les figures utilisent les poids restaurés à la meilleure époque de validation. Ne réduisez pas l’analyse à un score : la forme de la frontière est ici une information scientifique à part entière.


In [ ]:
torch.manual_seed(SEED + 10)
linear = train_classifier(nn.Linear(2, 2), X_train, y_train, X_val, y_val)
torch.manual_seed(SEED + 10)
mlp = train_classifier(make_mlp(), X_train, y_train, X_val, y_val)
experiments_B = {"Linéaire": linear, "MLP": mlp}
for name, result in experiments_B.items():
    metrics = evaluate_classifier(result["model"], X_val, y_val)
    print(f"{name:10s} | meilleure époque {result['best_epoch']:3d} | "
          f"perte val {metrics['loss']:.4f} | exactitude val {metrics['accuracy']:.3f}")
show_histories(experiments_B)
show_boundaries(experiments_B)


### Bilan du TP B — distinguer représentation et optimisation

1. Pourquoi une frontière affine ne peut-elle pas séparer exactement les quatre quadrants alternés ?
2. Pourquoi l’application d’un softmax avant `CrossEntropyLoss` serait-elle redondante et numériquement moins sûre ?
3. Quels indices permettraient de distinguer un modèle insuffisamment expressif d’un entraînement défaillant ?
4. Que garantit un théorème d’approximation universelle, et que ne garantit-il ni sur les données disponibles ni sur l’algorithme d’apprentissage ?


<a id="tp-c"></a>
## TP C — Du critère numérique à la modification des paramètres · 30 min

Nous disposons maintenant d’une famille de fonctions et d’une perte. Il reste à comprendre comment l’erreur finale est attribuée à chacun des paramètres. Cette partie ouvre la boîte de la rétropropagation : nous commencerons par une dérivation matricielle, nous la confronterons à `autograd`, puis nous écrirons une époque complète d’apprentissage.

### C1 — Dériver, vérifier, puis automatiser · 12 min

Pour $Z=XW+b$, $P=\operatorname{softmax}(Z)$ et une matrice indicatrice des cibles $Y$, l’entropie croisée moyenne vérifie

$$
\frac{\partial L}{\partial Z}=\frac{P-Y}{N},\qquad
\frac{\partial L}{\partial W}=X^\top\frac{P-Y}{N},\qquad
\frac{\partial L}{\partial b}=\sum_{i=1}^N\frac{P_i-Y_i}{N}.
$$

Complétez les deux derniers gradients en conservant la convention $(d,C)$ utilisée en A1. Vérifiez les dimensions avant les valeurs. Nous utilisons `float64` pour contrôler les dérivées ; cela ne signifie pas que tous les entraînements doivent être menés en double précision.

`gradcheck` compare l’autodifférentiation à des différences finies. Celles-ci servent de **contrôle local indépendant**, pas de méthode d’apprentissage. L’accord de plusieurs calculs est plus informatif que la seule observation d’une perte qui diminue.


In [ ]:
def manual_ce_grad(x, w, b, y):
    probabilities = (x @ w + b).softmax(dim=1)
    one_hot = F.one_hot(y, num_classes=w.shape[1]).to(x.dtype)
    dz = (probabilities - one_hot) / len(x)
    # TODO C1 : calculer dw et db, puis les renvoyer.
    raise NotImplementedError("C1 : gradient matriciel et somme sur le mini-lot")


In [ ]:
g = torch.Generator().manual_seed(SEED + 20)
xg = torch.randn(5, 2, generator=g, dtype=torch.float64)
wg = torch.randn(2, 3, generator=g, dtype=torch.float64, requires_grad=True)
bg = torch.randn(3, generator=g, dtype=torch.float64, requires_grad=True)
yg = torch.tensor([0, 1, 2, 0, 1], dtype=torch.long)
lg = F.cross_entropy(xg @ wg + bg, yg)
dw_auto, db_auto = torch.autograd.grad(lg, (wg, bg))
dw_manual, db_manual = manual_ce_grad(xg, wg.detach(), bg.detach(), yg)
print("Écart maximal sur W :", float((dw_manual - dw_auto).abs().max()))
print("Écart maximal sur b :", float((db_manual - db_auto).abs().max()))
assert torch.allclose(dw_manual, dw_auto, atol=1e-10, rtol=1e-8)
assert torch.allclose(db_manual, db_auto, atol=1e-10, rtol=1e-8)
passed = torch.autograd.gradcheck(lambda w, b: F.cross_entropy(xg @ w + b, yg),
                                 (wg, bg), eps=1e-6, atol=1e-5, rtol=1e-3)
print("Vérification par différences finies :", passed)


### C1 bis — L’accumulation des gradients est un choix algorithmique · 3 min

Prédisez les trois valeurs qui seront affichées avant d’exécuter la cellule. Dans PyTorch, `backward()` **ajoute** sa contribution au contenu de `.grad` ; il ne remplace pas automatiquement le gradient précédent. Nous reconstruisons ici le graphe à chaque appel, de sorte que `retain_graph=True` n’est pas nécessaire.

Cette accumulation est utile lorsqu’elle est intentionnelle — par exemple pour simuler un lot plus grand — mais elle change silencieusement l’algorithme si l’on oublie de remettre les gradients à zéro. `zero_grad` n’est donc pas une opération de ménage : il fait partie de la définition de la mise à jour.


In [ ]:
a = torch.tensor(2., requires_grad=True)
(a * a).backward()
first = a.grad.item()
(a * a).backward()
accumulated = a.grad.item()
a.grad = None
(a * a).backward()
after_reset = a.grad.item()
print("Premier backward / second sans effacement / après effacement :", first, accumulated, after_reset)
assert (first, accumulated, after_reset) == (4., 8., 4.)


### C2 — Écrire une époque d’apprentissage complète · 10 min

Complétez la boucle dans l’ordre logique suivant :

1. effacer les gradients de la mise à jour précédente ;
2. calculer les logits ;
3. construire la perte ;
4. propager ses dérivées ;
5. demander à l’optimiseur de modifier les paramètres.

Lancez ensuite 30 époques. Le modèle de cette partie sert à vérifier la mécanique ; il **ne s’ajoute pas** aux candidats du TP D. Le résultat attendu n’est pas un score spectaculaire, mais une boucle dont chaque ligne peut être reliée à une opération mathématique clairement identifiée.


In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, total_examples = 0.0, 0
    for xb, yb in loader:
        # TODO C2 : les quatre étapes de l'apprentissage ; définir la variable loss.
        raise NotImplementedError("C2 : zéro → prédiction/perte → backward → step")
        total_loss += loss.detach().item() * len(xb)
        total_examples += len(xb)
    return total_loss / total_examples


In [ ]:
torch.manual_seed(SEED + 30)
model_C = make_mlp()
loader_C = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED + 31))
optimizer_C = torch.optim.AdamW(model_C.parameters(), lr=.01, weight_decay=0.)
losses_C = [train_one_epoch(model_C, loader_C, optimizer_C) for _ in range(30)]
assert np.isfinite(losses_C).all()
print("Perte moyenne rencontrée dans les mini-lots :", round(losses_C[0], 4), "→", round(losses_C[-1], 4))
print("Validation après 30 époques :", evaluate_classifier(model_C, X_val, y_val))
print("Cette perte de mini-lots agrège des paramètres successifs ; elle diffère d'une perte recalculée en fin d'époque.")


### Bilan du TP C — séparer dérivation, état du modèle et mise à jour

1. Quelle instruction calcule les gradients, et quelle instruction modifie les paramètres ?
2. `model.eval()` interdit-il le calcul d’un gradient ? Expliquez ce qu’il change réellement.
3. `torch.no_grad()` désactive-t-il le dropout ? Pourquoi ces deux mécanismes ne sont-ils pas interchangeables ?
4. Pourquoi pondère-t-on la perte de chaque mini-lot par `len(xb)` avant de calculer la moyenne d’une époque ?


<a id="tp-d"></a>
## TP D — De l’ajustement à une conclusion défendable · 25 min

Une faible perte d’entraînement prouve seulement que le modèle s’est adapté aux observations qui ont influencé ses paramètres. Elle ne dit pas encore quelle configuration il faut retenir ni ce que cette configuration fera sur de nouvelles observations. Nous allons donc rendre explicite la différence entre **entraîner**, **sélectionner** et **évaluer**.

### Deux expériences contrôlées · 10 min

Nous entraînons le même MLP pendant 350 époques avec deux valeurs annoncées à l’avance pour `weight_decay` : 0 et 0.02. L’initialisation, le taux d’apprentissage, l’ordre des mini-lots et le budget restent identiques. AdamW applique une **décroissance découplée des poids** ; pour un optimiseur adaptatif, cette opération n’est pas équivalente à l’ajout naïf d’une pénalité quadratique au gradient. Afin de garder le code visible, la décroissance s’applique ici à tous les paramètres, biais compris.

Avant l’exécution, dessinez plusieurs scénarios plausibles pour les courbes. Une régularisation peut améliorer la validation, rester sans effet mesurable ou détériorer l’optimisation. L’expérience comporte une seule graine : elle fournit un exemple reproductible, pas une loi générale sur les MLP.


In [ ]:
experiments_D = {}
for wd in (0.0, 0.02):
    torch.manual_seed(SEED + 40)  # même initialisation dans les deux expériences
    result = train_classifier(make_mlp(), X_train, y_train, X_val, y_val,
                              epochs=350, lr=.01, weight_decay=wd, seed=SEED + 41)
    name = f"MLP long, wd={wd:g}"
    experiments_D[name] = result
    print(f"{name:24s} | meilleure époque {result['best_epoch']:3d} | perte val {result['best_val_loss']:.4f}")
show_histories(experiments_D, "regularization_learning")


### D1 — Sélectionner sur la validation et figer le bon état · 7 min

Parmi les quatre candidats annoncés avant l’expérience, retenez celui dont la **meilleure perte de validation** est la plus faible. La fonction fournie a déjà restauré les poids correspondant à cette époque. Ne choisissez donc ni le dernier état par habitude, ni une nouvelle variante inspirée par un résultat de test encore à venir.

**À expliquer.** Pourquoi `best_state = model.state_dict()` sans copie profonde ne suffit-il pas toujours à figer un checkpoint ? Pourquoi la validation participe-t-elle indirectement au processus d’apprentissage, même si aucun `backward()` n’est calculé sur ses observations ? La réponse doit faire apparaître la notion de **sélection**.


In [ ]:
candidates = {**experiments_B, **experiments_D}
# TODO D1 : renvoyer le nom du candidat de plus petite best_val_loss.
def choose_by_validation(candidates):
    raise NotImplementedError("D1 : une sélection fondée exclusivement sur la validation")

chosen_name = choose_by_validation(candidates)
chosen_result = candidates[chosen_name]
chosen_model = chosen_result["model"]
print("Choix verrouillé :", chosen_name, "| époque", chosen_result["best_epoch"])


### D2 — Ouvrir le test une seule fois · 3 min

Le modèle étant choisi, nous pouvons évaluer la procédure finale sur les 180 observations de test. Elles reçoivent la transformation déterminée sur le train ; aucune statistique n’est réestimée.

L’intervalle de Wilson à 95 % affiché ci-dessous décrit l’incertitude binomiale associée à l’exactitude de ce prédicteur fixé, sous l’hypothèse d’observations indépendantes. Il ne mesure ni la variabilité due à une autre initialisation, ni l’incertitude liée au choix de l’architecture, ni un éventuel décalage de distribution.

Dès que ce résultat influence un nouveau réglage, ce jeu cesse d’être un test indépendant pour cette étude. Il serait méthodologiquement incorrect de relancer plusieurs graines puis de ne conserver que celle qui donne le nombre le plus flatteur.


In [ ]:
def wilson_interval(k, n, z=1.959963984540054):
    p = k / n
    denominator = 1 + z*z/n
    center = (p + z*z/(2*n)) / denominator
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denominator
    return center - radius, center + radius

if globals().get("TEST_ALREADY_OPENED", False):
    print("Test déjà ouvert : conserver le résultat initial ci-dessous, sans nouveau réglage.")
    print(final_report)
else:
    X_test_raw, y_test = TEST_SCELLE
    X_test = (X_test_raw - mean_train) / std_train
    chosen_model.eval()
    with torch.no_grad():
        test_logits = chosen_model(X_test)
        test_loss = float(loss_fn(test_logits, y_test))
        correct = int((test_logits.argmax(1) == y_test).sum())
    low, high = wilson_interval(correct, len(y_test))
    final_report = {"model": chosen_name, "best_epoch": chosen_result["best_epoch"],
                    "test_loss": test_loss, "test_accuracy": correct / len(y_test),
                    "wilson_95": (low, high), "n_test": len(y_test)}
    TEST_ALREADY_OPENED = True
    print(final_report)


### Bilan du TP D — limiter la portée de ce que l’on affirme

1. Quelle époque aurait été choisie si l’on avait minimisé uniquement la perte d’entraînement ? En quoi cette règle répond-elle à une autre question ?
2. L’expérience permet-elle d’affirmer que le `weight_decay` améliore toujours un MLP ? Formulez une conclusion proportionnée aux observations.
3. Pourquoi l’intervalle d’exactitude affiché ne suffit-il pas à établir la supériorité générale d’une architecture ?


<a id="tp-e"></a>
## TP E — Quand l’information prend la forme d’une loi physique · 30 min

Jusqu’ici, chaque contrainte d’apprentissage provenait d’un couple entrée–étiquette. Nous allons maintenant conserver le même mécanisme différentiable, mais changer la nature de l’information : une équation différentielle et des conditions initiales vont définir ce qu’est une prédiction acceptable.

### Une expérience assez simple pour être vérifiée indépendamment

Nous cherchons la solution de

$$
\ddot x(t)+2\gamma\dot x(t)+\omega_0^2x(t)=0,
\qquad x(0)=1,\quad \dot x(0)=0,
$$

avec $\gamma=0.3\;\mathrm{s}^{-1}$, $\omega_0=2\;\mathrm{s}^{-1}$ et $0\le t\le T=3\;\mathrm{s}$. Dans le régime sous-amorti, $\omega_d=\sqrt{\omega_0^2-\gamma^2}$ et

$$
x_\star(t)=e^{-\gamma t}\left[x_0\cos(\omega_dt)
+\frac{v_0+\gamma x_0}{\omega_d}\sin(\omega_dt)\right].
$$

Le réseau reçoit la variable sans dimension $s=t/T\in[0,1]$. La règle de chaîne impose donc $\dot x=u_s/T$ et $\ddot x=u_{ss}/T^2$ : normaliser l’entrée sans transformer les dérivées changerait le problème physique.

Pour satisfaire exactement les conditions initiales, nous paramétrons

$$u_\theta(s)=x_0+Tv_0s+s^2N_\theta(s).$$

La perte repose sur le résidu normalisé

$$
\widetilde r_\theta(s)=\frac{u_{ss}}{\omega_0^2T^2}
+\frac{2\gamma u_s}{\omega_0^2T}+u_\theta(s).
$$

Les coefficients physiques sont connus et ne sont pas optimisés. La solution analytique et le solveur classique seront utilisés **uniquement pour le contrôle final**, jamais comme cibles d’apprentissage.


In [ ]:
GAMMA, OMEGA0, T_FINAL = 0.3, 2.0, 3.0
X0, V0 = 1.0, 0.0
PINN_DTYPE = torch.float64

def exact_solution(t):
    omega_d = math.sqrt(OMEGA0**2 - GAMMA**2)
    return torch.exp(-GAMMA * t) * (X0 * torch.cos(omega_d * t)
           + (V0 + GAMMA * X0) / omega_d * torch.sin(omega_d * t))

class OscillatorPINN(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, width), nn.Tanh(),
                                 nn.Linear(width, width), nn.Tanh(), nn.Linear(width, 1))
    def forward(self, s):
        return X0 + T_FINAL * V0 * s + s.square() * self.net(s)

torch.manual_seed(SEED + 50)
pinn = OscillatorPINN().to(dtype=PINN_DTYPE, device=DEVICE)
s_collocation = torch.linspace(0., 1., 80, dtype=PINN_DTYPE).reshape(-1, 1)
print("Paramètres du PINN :", sum(p.numel() for p in pinn.parameters()))
print("Nombre de points de collocation :", len(s_collocation))


### E1 — Différencier le réseau par rapport à son entrée · 10 min

Complétez deux appels successifs à `torch.autograd.grad` afin d’obtenir $u_s$ puis $u_{ss}$, et construisez le résidu en conservant les facteurs $T$ et $\omega_0$. Pour chaque appel, fournissez `grad_outputs=torch.ones_like(...)` et `create_graph=True`. Ce dernier argument est nécessaire parce que les dérivées par rapport à l’entrée font elles-mêmes partie du calcul que nous devrons ensuite différencier par rapport aux paramètres.

Le réseau traite ici chaque ligne indépendamment. La dérivée de la somme des sorties par rapport au lot d’entrées coïncide donc avec les dérivées point par point attendues. Cette commodité ne se transpose pas automatiquement à un modèle qui mélangerait les observations du lot.

`Tanh` possède les dérivées régulières nécessaires. Un MLP ReLU ordinaire est affine par morceaux : sa dérivée seconde classique est nulle presque partout et indéfinie aux ruptures. Il ne convient donc pas directement à cette formulation forte d’une EDO du second ordre.


In [ ]:
def derivatives_and_residual(model, s_values):
    s = s_values.detach().clone().requires_grad_(True)
    u = model(s)
    # TODO E1 : du = du/ds puis d2u = d²u/ds², avec create_graph=True.
    # TODO E1 : résidu physique divisé par OMEGA0**2.
    raise NotImplementedError("E1 : deux dérivées automatiques et règle de la chaîne")
    return u, du, d2u, residual


In [ ]:
class ExactReference(nn.Module):
    def forward(self, s):
        return exact_solution(T_FINAL * s)

_, _, _, exact_residual = derivatives_and_residual(ExactReference(), s_collocation)
assert float(exact_residual.detach().abs().max()) < 1e-10
u0, du0, _, _ = derivatives_and_residual(pinn, torch.zeros(1, 1, dtype=PINN_DTYPE))
assert abs(float(u0.detach()) - X0) < 1e-12
assert abs(float(du0.detach()) / T_FINAL - V0) < 1e-12
print("Résidu normalisé de la solution exacte :", float(exact_residual.detach().abs().max()))
print("Conditions initiales imposées avant entraînement :", float(u0.detach()), float(du0.detach()) / T_FINAL)


### E2 — Optimiser le résidu sans confondre méthode et garantie · 12 min

La procédure d’optimisation est fournie. Adam effectue une première phase de descente ; L-BFGS affine ensuite la solution sur un ensemble fixe de points de collocation. Cette combinaison est utile sur ce petit problème, mais elle ne constitue pas une recette universelle pour les PINNs. L-BFGS peut appeler plusieurs fois sa fermeture au cours d’une même itération ; nombre d’itérations et nombre d’évaluations de la perte ne sont donc pas identiques.

Le critère ne contient que le résidu différentiel, car la paramétrisation impose déjà les conditions initiales. Sans ces conditions, la fonction nulle satisferait elle aussi l’équation : une loi différentielle seule ne sélectionne pas nécessairement l’expérience physique que l’on veut représenter.


In [ ]:
def fit_pinn(model, points, adam_steps=1200, lbfgs_iterations=200):
    trace = []
    adam = torch.optim.Adam(model.parameters(), lr=0.002)
    model.train()
    for step in range(adam_steps):
        adam.zero_grad(set_to_none=True)
        _, _, _, residual = derivatives_and_residual(model, points)
        loss = residual.square().mean()
        loss.backward()
        adam.step()
        trace.append(float(loss.detach()))
    lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0,
                             max_iter=lbfgs_iterations, max_eval=350,
                             tolerance_grad=1e-10, tolerance_change=1e-12,
                             line_search_fn="strong_wolfe")
    closure_values = []
    def closure():
        lbfgs.zero_grad(set_to_none=True)
        _, _, _, residual = derivatives_and_residual(model, points)
        loss = residual.square().mean()
        loss.backward()
        closure_values.append(float(loss.detach()))
        return loss
    lbfgs.step(closure)
    model.eval()
    return trace, closure_values

pinn_start = time.perf_counter()
pinn_trace, lbfgs_trace = fit_pinn(pinn, s_collocation)
print(f"Entraînement PINN : {time.perf_counter() - pinn_start:.1f} s sur cet environnement")
print("Évaluations Adam / fermeture L-BFGS :", len(pinn_trace), len(lbfgs_trace))
print("Dernière perte Adam / dernière perte L-BFGS :", pinn_trace[-1], lbfgs_trace[-1])


### E3 — Contrôler la solution ailleurs que là où elle a été ajustée · 8 min

Nous évaluons le réseau sur **400 points intercalés, distincts des points de collocation**, sans réentraîner après inspection. Trois contrôles répondent à trois questions différentes :

1. la trajectoire est-elle proche d’une référence indépendante ?
2. l’équation est-elle satisfaite entre les points utilisés par l’optimisation ?
3. les conditions initiales sont-elles effectivement respectées ?

La racine de la moyenne des carrés du résidu physique s’exprime en unité de déplacement par seconde carrée, tandis que le résidu normalisé possède l’unité du déplacement. Une petite moyenne quadratique sur quelques points ne constitue pas, à elle seule, une borne uniforme de l’erreur de solution.

Pour calculer uniquement les valeurs de la fonction, `torch.no_grad()` convient. Pour contrôler un résidu différentiel, le suivi des gradients par rapport à l’entrée doit rester actif, même si `model.eval()` a été appelé. Cette distinction est fondamentale : le mode d’évaluation des couches et l’activation de l’autodifférentiation sont deux dimensions indépendantes du calcul.


In [ ]:
s_check = ((torch.arange(400, dtype=PINN_DTYPE) + .5) / 400).reshape(-1, 1)
assert torch.cdist(s_check, s_collocation).min() > 1e-8
# Pas de no_grad ici : les dérivées par rapport à l'entrée sont nécessaires.
u_check, _, _, normalized_residual = derivatives_and_residual(pinn, s_check)
t_check = T_FINAL * s_check
reference = exact_solution(t_check)
relative_l2 = float((torch.linalg.vector_norm(u_check - reference)
                     / torch.linalg.vector_norm(reference)).detach())
max_error = float((u_check - reference).detach().abs().max())
residual_rms = float((OMEGA0**2 * normalized_residual.detach()).square().mean().sqrt())
u0, du0, _, _ = derivatives_and_residual(pinn, torch.zeros(1, 1, dtype=PINN_DTYPE))
pinn_report = {"relative_l2": relative_l2, "max_error": max_error,
               "physical_residual_rms": residual_rms,
               "initial_position_error": abs(float(u0.detach()) - X0),
               "initial_velocity_error": abs(float(du0.detach()) / T_FINAL - V0)}
print(pinn_report)
assert all(np.isfinite(v) for v in pinn_report.values())

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].plot(t_check.numpy(), reference.numpy(), color="black", label="Solution analytique")
axes[0].plot(t_check.numpy(), u_check.detach().numpy(), "--", label="PINN")
axes[0].set(xlabel="Temps (s)", ylabel="Déplacement", title="Solution hors collocation")
axes[0].legend(fontsize=8)
axes[1].plot(t_check.numpy(), (OMEGA0**2 * normalized_residual).detach().numpy())
axes[1].set(xlabel="Temps (s)", ylabel="Résidu physique", title="Équation contrôlée ailleurs")
axes[2].semilogy(np.arange(1, len(pinn_trace)+1), pinn_trace, label="Adam")
axes[2].semilogy(np.arange(len(pinn_trace)+1, len(pinn_trace)+len(lbfgs_trace)+1), lbfgs_trace, label="L-BFGS")
axes[2].set(xlabel="Évaluation de perte", ylabel="Moyenne du résidu normalisé²", title="Optimisation")
axes[2].legend(fontsize=8)
fig.tight_layout()
export_figure(fig, "pinn_controls")
plt.show()


In [ ]:
# Référence numérique classique : contrôle du PINN et de la formule analytique.
from scipy.integrate import solve_ivp
solver_start = time.perf_counter()
t_numpy = t_check[:, 0].numpy()
sol_ivp = solve_ivp(lambda t, z: [z[1], -2*GAMMA*z[1] - OMEGA0**2*z[0]],
                    (0., T_FINAL), [X0, V0], t_eval=t_numpy,
                    method="DOP853", rtol=1e-10, atol=1e-12)
assert sol_ivp.success
ivp_seconds = time.perf_counter() - solver_start
ivp_error = np.max(np.abs(sol_ivp.y[0] - reference[:, 0].numpy()))
print(f"solve_ivp : {ivp_seconds:.4f} s ; écart maximal à la formule analytique : {ivp_error:.3e}")
print("Ces chronométrages illustrent le cas présent ; ils ne constituent pas un benchmark général.")


### Bilan du TP E — préciser ce qui a réellement été appris

1. Pourquoi la sortie identiquement nulle est-elle exclue par notre paramétrisation ?
2. Quel facteur de conversion serait perdu si l’on confondait une dérivée par rapport à $s$ avec une dérivée par rapport à $t$ ?
3. Pourquoi `create_graph=True` est-il nécessaire pendant l’entraînement ?
4. Que démontre cette expérience contrôlée sur la construction d’un PINN, et que ne démontre-t-elle pas sur sa supériorité face aux méthodes numériques classiques ?


<a id="sortie"></a>
## Ticket de sortie · 5 min

Répondez sans lancer de nouveau calcul, puis confrontez les réponses avec le binôme voisin. Le but est de reformuler la séance comme une chaîne de décisions, et non comme une succession de commandes PyTorch.

1. La perte d’entraînement est faible mais la perte de validation reste forte. Citez deux hypothèses de nature différente à examiner.
2. Une sortie de forme `(64, 2)` et des cibles de forme `(64,)` sont-elles compatibles avec l’entropie croisée ? Quel doit être le type de ces cibles ?
3. Un appel à `backward()` modifie-t-il directement les poids ?
4. Pourquoi un hyperparamètre choisi grâce au test interdit-il de présenter ce même test comme une évaluation indépendante ?
5. Dans le PINN, quels rôles distincts jouent l’équation différentielle et les conditions initiales ?
6. Que vous a appris le quartet d’Anscombe que ne pouvait pas montrer le seul tableau de statistiques ?

**Trace à conserver.** Une figure commentée du quartet d’Anscombe, une frontière de décision, le rapport de sélection/test, les trois contrôles du PINN et six phrases répondant au ticket. Chaque résultat doit pouvoir être expliqué sans invoquer le réseau ou le logiciel comme une autorité.


## Prolongements facultatifs — pour transformer le TP en véritable étude

Ces activités dépassent les cinq heures. Elles sont proposées pour prolonger la méthode expérimentale, non pour accumuler des architectures.

**1. Mesurer la variabilité d’une comparaison.** Répéter le protocole complet sur cinq graines avec un budget fixé, puis résumer moyenne et dispersion des pertes de validation. Distinguer la variabilité du réentraînement de l’incertitude sur un test fini. Ne pas recycler les 180 observations déjà ouvertes pour poursuivre les réglages.

**2. Inspecter le comportement des couches.** Introduire un dropout entre les couches cachées et comparer des prédictions répétées en modes `train` et `eval`, avec puis sans enregistrement des gradients. Prévoir les quatre cas avant de les exécuter. La dispersion induite par dropout ne constitue pas automatiquement une incertitude probabiliste calibrée.

**3. Dégrader le PINN de manière contrôlée.** Allonger l’horizon, raréfier les points de collocation ou remplacer `Tanh` par ReLU. Repartir d’une nouvelle instance et conserver une expérience de référence. Comparer conjointement l’erreur de trajectoire et le résidu sur une grille indépendante. Avec l’enveloppe $s^2$, la sortie complète peut conserver de la courbure, mais l’autodifférentiation point par point ne représente pas les contributions singulières aux ruptures d’un réseau ReLU : la formulation forte demeure délicate.

**4. Passer à un problème inverse.** Rendre $\gamma$ et $\omega_0$ inconnus et ajouter des mesures bruitées de $x(t)$. Construire une perte combinant données et dynamique, puis étudier identifiabilité, sensibilité et robustesse à l’initialisation. Une contrainte de positivité peut être imposée par `softplus`, mais elle ne garantit pas que les observations contiennent assez d’information pour identifier les paramètres.

**5. Revenir au quartet d’Anscombe.** Réentraîner une régression après retrait successif d’un point. Comparer en particulier le troisième et le quatrième jeu : un point atypique en $y$ et un point de fort levier en $x$ n’influencent pas la droite de la même manière. Relier cette expérience aux notions de robustesse, de diagnostic des résidus et de décalage de distribution.


<a id="optimisateurs"></a>
## Complément expérimental — SGD, RMSprop, Adam et AdamW · hors séance

Ce complément prolonge le chapitre du cours consacré aux optimiseurs ; il ne constitue pas un sixième TP obligatoire. La même discipline que pour les données s’applique : annoncer la grandeur que l’on compare, modifier un seul mécanisme à la fois et résister à la tentation de désigner un vainqueur à partir d’une trajectoire isolée.

### O1 — Pourquoi corriger le démarrage d’Adam ?

Avec $m_0=v_0=0$, le premier gradient donne $m_1=(1-\beta_1)g_1$ et $v_1=(1-\beta_2)g_1^2$. Les estimateurs corrigés sont $\widehat m_1=g_1$ et $\widehat v_1=g_1^2$. L’expression « correction du biais » désigne ici la masse manquante des moyennes exponentielles initialisées à zéro ; elle ne transforme pas la direction complète d’Adam en estimateur sans biais d’un gradient stationnaire idéal.

**Question.** Pour $g_1=2$, $\beta_1=0.9$, $\beta_2=0.999$ et $\alpha=0.1$, calculez la mise à jour avec et sans correction, d’abord en négligeant $\varepsilon$, puis vérifiez numériquement. RMSprop usuel conserve lui aussi une moyenne des carrés, mais sans cette correction de démarrage.


In [ ]:
g1 = torch.tensor(2., dtype=torch.float64)
beta1, beta2, alpha, eps = .9, .999, .1, 1e-8
m1, v1 = (1-beta1)*g1, (1-beta2)*g1.square()
m1_hat, v1_hat = m1/(1-beta1), v1/(1-beta2)
uncorrected_step = alpha * m1 / (v1.sqrt() + eps)
corrected_step = alpha * m1_hat / (v1_hat.sqrt() + eps)
print(f"m1={m1:.4f}, v1={v1:.4f}, m1 corrigé={m1_hat:.4f}, v1 corrigé={v1_hat:.4f}")
print(f"Quantité soustraite : sans correction {uncorrected_step:.6f} ; Adam corrigé {corrected_step:.6f}")


### O2 — Une vallée étroite éclaire la géométrie, pas la généralisation

Nous minimisons la quadratique $f(\theta)=\tfrac12\theta^\top H\theta$, dont les valeurs propres sont 1 et 50. Une rotation de 30° rend les directions propres obliques aux axes des paramètres. Tous les algorithmes partent de $(3,3)$ et disposent de 250 mises à jour, mais leurs taux sont explicitement adaptés à la nature de leur règle : imposer la même valeur numérique à des mises à jour différentes ne constitue pas automatiquement une comparaison équitable.

SGD utilise un pas 0.03, inférieur à $2/\lambda_{\max}=0.04$ ; RMSprop utilise 0.07 avec `alpha=0.9`, sans momentum ; Adam et AdamW utilisent 0.08 avec $(\beta_1,\beta_2)=(0.9,0.999)$. AdamW ajoute un `weight_decay` de 0.03 et n’effectue donc pas exactement la même optimisation. La grandeur représentée reste toutefois la **même perte de données**.

**Questions.** Que limite un préconditionnement diagonal lorsque les directions principales sont tournées ? Une descente rapide sur cette quadratique prédit-elle la généralisation d’un réseau ? Pourquoi un calendrier de taux pourrait-il modifier l’allure des trajectoires ?


In [ ]:
angle = math.pi / 6
rotation = torch.tensor([[math.cos(angle), -math.sin(angle)],
                          [math.sin(angle), math.cos(angle)]], dtype=torch.float64)
H = rotation @ torch.diag(torch.tensor([1., 50.], dtype=torch.float64)) @ rotation.T

def quadratic(theta):
    return .5 * theta @ H @ theta

optimizer_factories = {
    "SGD, lr=.03": lambda params: torch.optim.SGD(params, lr=.03),
    "RMSprop, lr=.07": lambda params: torch.optim.RMSprop(params, lr=.07, alpha=.9, eps=1e-8, momentum=0.),
    "Adam, lr=.08": lambda params: torch.optim.Adam(params, lr=.08, betas=(.9, .999), eps=1e-8),
    "AdamW, lr=.08, wd=.03": lambda params: torch.optim.AdamW(params, lr=.08, betas=(.9, .999), eps=1e-8, weight_decay=.03),
}
optimizer_results = {}
for name, factory in optimizer_factories.items():
    theta = nn.Parameter(torch.tensor([3., 3.], dtype=torch.float64))
    optimizer = factory([theta])
    trajectory = [theta.detach().clone()]
    values = [float(quadratic(theta).detach())]
    for step in range(250):
        optimizer.zero_grad(set_to_none=True)
        value = quadratic(theta)
        value.backward()
        optimizer.step()
        trajectory.append(theta.detach().clone())
        values.append(float(quadratic(theta).detach()))
    optimizer_results[name] = {"trajectory": torch.stack(trajectory).numpy(), "loss": np.array(values)}
    print(f"{name:28s} | perte finale {values[-1]:.6g}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
grid = np.linspace(-1.5, 4., 220)
qx, qy = np.meshgrid(grid, grid)
coords = np.stack([qx, qy], axis=-1)
qvalues = .5 * np.einsum("...i,ij,...j->...", coords, H.numpy(), coords)
axes[0].contour(qx, qy, qvalues, levels=[.1, 1, 5, 20, 80, 200], colors="0.8", linewidths=.8)
for name, result in optimizer_results.items():
    trajectory = result["trajectory"]
    axes[0].plot(trajectory[:, 0], trajectory[:, 1], label=name, linewidth=1.3)
    axes[1].semilogy(result["loss"], label=name)
axes[0].scatter([0], [0], marker="*", color="black", s=80)
axes[0].set(xlabel=r"$\theta_1$", ylabel=r"$\theta_2$", title="Trajectoires : vallée quadratique", aspect="equal")
axes[1].set(xlabel="Pas d'optimisation", ylabel="Perte de données", title="Même point initial, 250 pas")
axes[1].legend(fontsize=7)
axes[1].grid(alpha=.2)
fig.tight_layout()
export_figure(fig, "optimizer_trajectories")
plt.show()


### O3 — Pénalisation quadratique dans Adam et décroissance AdamW

Une pénalité $\lambda\lVert\theta\rVert^2/2$ ajoute $\lambda\theta$ au gradient **avant** la mise à jour des moments d’Adam. AdamW calcule les moments à partir du gradient de données, puis applique séparément la contraction $\theta\leftarrow(1-\alpha\lambda)\theta$. La décroissance découplée et la correction du démarrage répondent donc à deux questions entièrement différentes.

Le premier pas ci-dessous part de $\theta=(1,2)$, avec $g=(-0.05,0.5)$, $\alpha=0.1$ et $\lambda=0.1$. Dans la version couplée, le premier gradient change même de signe. Nous injectons ces gradients connus afin d’isoler les règles de mise à jour ; il ne s’agit pas d’un nouvel entraînement sur le jeu XOR.


In [ ]:
from IPython.display import display, Markdown
initial = torch.tensor([1., 2.], dtype=torch.float64)
data_gradient = torch.tensor([-.05, .5], dtype=torch.float64)
regularization = .1
rows = []
for method in ["Adam sans régularisation", "Adam + pénalité L2", "AdamW"]:
    theta = nn.Parameter(initial.clone())
    if method == "AdamW":
        opt = torch.optim.AdamW([theta], lr=.1, betas=(.9, .999), eps=1e-8, weight_decay=regularization)
        theta.grad = data_gradient.clone()
    else:
        opt = torch.optim.Adam([theta], lr=.1, betas=(.9, .999), eps=1e-8, weight_decay=0.)
        theta.grad = data_gradient.clone()
        if method == "Adam + pénalité L2":
            theta.grad += regularization * theta.detach()
    opt.step()
    rows.append((method, theta.detach().tolist()))
lines = ["| Méthode | θ₁ après un pas | θ₂ après un pas |", "|---|---:|---:|"]
lines.extend(f"| {name} | {value[0]:.6f} | {value[1]:.6f} |" for name, value in rows)
display(Markdown("\n".join(lines)))


### Interprétation du complément

Expliquez, avec vos propres mots, la différence entre :

- la mémoire du gradient ;
- la mémoire du carré du gradient ;
- la correction du démarrage des moyennes exponentielles ;
- la décroissance des poids.

Citez ensuite au moins quatre éléments à consigner pour rendre une comparaison reproductible : taux d’apprentissage, coefficients de mémoire, $\varepsilon$, décroissance, taille des lots, ordre des données, initialisation, budget et calendrier de taux sont des exemples possibles.


## Sources et documentation pour poursuivre

Le polycopié joint donne la bibliographie scientifique complète. Les références suivantes sont directement liées aux opérations réalisées dans ce notebook.

- F. J. Anscombe, [*Graphs in Statistical Analysis*](https://doi.org/10.1080/00031305.1973.10478966), *The American Statistician* 27(1), 1973 : quatre jeux de données aux statistiques proches, conçus pour montrer la nécessité de représenter les observations.
- PyTorch, [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html) et [broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html) : formes, indexation et diffusion des dimensions.
- PyTorch, [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) : logits et format des cibles.
- PyTorch, [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html) : graphe de calcul, accumulation des gradients et mécanismes d’évaluation.
- PyTorch, [gradcheck](https://docs.pytorch.org/docs/stable/generated/torch.autograd.gradcheck.html) : contrôle numérique des dérivées en double précision.
- D. P. Kingma et J. Ba, [*Adam: A Method for Stochastic Optimization*](https://arxiv.org/abs/1412.6980), ICLR 2015.
- I. Loshchilov et F. Hutter, [*Decoupled Weight Decay Regularization*](https://openreview.net/forum?id=Bkg6RiCqY7), ICLR 2019.
- M. Raissi, P. Perdikaris et G. E. Karniadakis, [*Physics-informed neural networks*](https://doi.org/10.1016/j.jcp.2018.10.045), *Journal of Computational Physics* 378, 2019.
- A. S. Krishnapriyan et al., [*Characterizing possible failure modes in physics-informed neural networks*](https://proceedings.neurips.cc/paper/2021/hash/df438e5206f31600e6ae4af72f2725f1-Abstract.html), NeurIPS 2021.
- S. Wang et al., [*An Expert’s Guide to Training Physics-informed Neural Networks*](https://arxiv.org/abs/2308.08468), 2023.

Les API PyTorch sont documentées en ligne et peuvent évoluer. Pour une expérience reproductible, consignez les versions effectivement utilisées, les graines, le matériel, le budget et les options des optimiseurs.


In [ ]:
print(f"Durée totale des calculs, hors interactions : {time.perf_counter() - NOTEBOOK_START:.1f} s")
